# Transport a packet and detect changed bytes

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 fixtures.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.


## Start here

A codec changes representation while preserving a tree. A digest detects a changed record only when the receiver recomputes it and checks which record the digest names.

## Exact model

decode(encode(tree)) = tree; the DAG envelope binds node content to SHA-256 keys.

All numerical fixtures use integers or Fraction. A passing run means the named finite checks passed.


In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "education/scripts/build_system_integration.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from education.scripts.system_integration_support import (
    expect_error,
)


def load_source(name, relative):
    spec = importlib.util.spec_from_file_location(name, ROOT / relative)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

## Worked computation

Run the cell, then compare the output with the exact model above.


In [ ]:
from pfp.domain.tower import wire

tree = wire.parse_wire("op<<0><1>op<<0><1>>>")
for encode, decode in [
    (wire.emit_compact, wire.parse_compact),
    (wire.emit_postfix, wire.parse_postfix),
    (wire.emit_tagged_json, wire.parse_tagged_json),
    (wire.emit_dag, wire.parse_dag),
]:
    assert decode(encode(tree)) == tree
envelope = wire.emit_dag(tree)
assert wire.parse_dag(envelope) == tree
print("Four codecs preserve the declared tree.")

## Exercise and adversarial check

Change one atom in the DAG node table while keeping its old digest key. Expect a refusal.

Predict the result before running the reference check below. Each assertion checks a specific value or failure mode.


In [ ]:
tampered = json.loads(envelope)
atom_key = next(key for key, value in tampered["nodes"].items() if value.get("k") == "a")
tampered["nodes"][atom_key]["v"] = "11"
expect_error(
    wire.WireError,
    lambda: wire.parse_dag(json.dumps(tampered, sort_keys=True, separators=(",", ":"))),
)
assert wire.emit_dag(tree) == envelope
print("Changed node content cannot reuse the old content digest.")

## Evidence boundary

The DAG serializer has a declared restricted JSON form. Sorted JSON alone is not a general RFC 8785 implementation.

## Source route

- [symbol_game/domain/tower/wire.py](../../../../../symbol_game/domain/tower/wire.py)
- [research/mssn_sn_tower/wire/canonicalization.md](../../../../../research/mssn_sn_tower/wire/canonicalization.md)

Return to the [novice integration route](../verified_route.md) and [curriculum overview](../curriculum_25_100.md).
